In [1]:
import os
import re
import sys
import json
import pickle
import importlib
import itertools
import subprocess
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.stats import pearsonr

sys.path.append('/host/verges/tank/data/daniel/00_commonUtils/00_code/genUtils/')
import t1, gen

import visUtils as vis
import projectUtils as prjUtils

# TODO. AP coordinates, subtract all values by min

In [2]:
# parameters

AP_coords_pth = "/host/verges/tank/data/daniel/04_inVivoHistology/code/resources/coordinates/coords-AntPost_sub-PNC027_ses-a1_hemi-L_ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_13Mar2026-1810_mask-mTemp.func.gii"
AN_coords_pth = "/host/verges/tank/data/daniel/04_inVivoHistology/code/resources/coordinates/coords-AlloNeo_sub-PNC027_ses-a1_hemi-L_ctxSurf-fsLR-32k_ctxLbl-pial_hippSurf-den-0p5mm_hippLbl-inner_stitched_13Mar2026-1810_mask-mTemp.func.gii"


dirs_project = {
    'dir_root': '/host/verges/tank/data/daniel/04_inVivoHistology',
    'dir_data': 'data/',
    'dir_out': 'outputs/',
    'demo_dfs': 'demo_dfs',
    'figs': 'figs',
    'pathLists': 'pathLists',
}

analysis_params = {
    
    'time': '09Apr2026-1204', # for file naming
    
    'verbose': True,
    'qMap_names':["T1map"], # names of qMAP of interest
    
    'ctrl_grp':"CTRL_match", # reference to Z score on
    'test_grps': ["TLE_L", "TLE_R", "TLE_LR"], # groups to compare to reference for Z score (excluding the ctrl_grp)
    
    'icFlip': True,
    'ipsiTo': 'L',
    'contraTo': 'R',
    
    'nSurfs':16,
    'smoothing':["0", "0p5"],

    'equiVol_str': "equivol",
}

demo_dicts_json_pth = os.path.join(dirs_project['dir_root'], dirs_project['dir_out'], f"grp_dicts_{analysis_params['time']}.json" )
print(f"\nLoading dict_list ({gen.fmt_file_size(demo_dicts_json_pth)}): {demo_dicts_json_pth}")
with open(demo_dicts_json_pth, 'r') as f:
    dict_list = json.load(f)

visualization_params = {}

study_dicts = [
    { # MICS
        'studyName': 'MICs',
        'studyDescrip': '3T',
        'dir_root': '/data/mica3/BIDS_MICs/',
        'dir_raw': 'rawdata/',
        'dir_deriv': 'derivatives/',
        'dir_fs': 'freesurfer/',
        'dir_mp': 'micapipe_v0.2.0/',
        'dir_hu': 'hippunfold_v1.3.0/hippunfold/', # update to v2?
    },
    { # PNI
        'studyName': 'PNI',
        'studyDescrip': '7T',
        'dir_root': '/data/mica3/BIDS_PNI/',
        'dir_raw': 'rawdata/',
        'dir_deriv': 'derivatives/',
        'dir_fs': 'fastsurfer/',
        'dir_mp': 'micapipe_v0.2.0/',
        'dir_hu': 'hippunfold_v1.3.0/hippunfold/', # update to v2?

        'dir_root_pilot': '/host/verges/tank/data/MICA-7T-pilot/',
        'dir_deriv_pilot': 'derivatives/',
        'dir_mp_pilot': 'micapipe/',
    }
]

path_list_dir = os.path.join(dirs_project['dir_root'], dirs_project['dir_out'], 'group_maps', analysis_params['time'], dirs_project['pathLists'])



Loading dict_list (17KB): /host/verges/tank/data/daniel/04_inVivoHistology/outputs/grp_dicts_09Apr2026-1204.json


In [ ]:
importlib.reload(vis)
surface_ex = "/host/verges/tank/data/daniel/04_inVivoHistology/data/PNI/sub-PNC045_ses-a1/surfs/sub-PNC045_ses-a1_hemi-L_09Apr2026-1204_equivol-2of16.surf.gii"
an_coords = nib.load(AN_coords_pth).darrays[0].data
ap_coords = nib.load(AP_coords_pth).darrays[0].data
faces = nib.load(surface_ex).darrays[1].data
vertices = nib.load(surface_ex).darrays[0].data
_ = vis.show_mesh_map(vertices, faces, values = an_coords, title = f"Allo-Neo coordinates", label_vals = "Distance from dentate gyrus, (mm)")
_ = vis.show_mesh_map(vertices, faces, values = ap_coords, title = f"Ant-Post coordinates", label_vals = "MRI's y-axis, mm")

Widget(value='<iframe src="http://localhost:37421/index.html?ui=P_0x7fb8e19d5110_3&reconnect=auto" class="pyvi…

Widget(value='<iframe src="http://localhost:37421/index.html?ui=P_0x7fb8f23f7190_4&reconnect=auto" class="pyvi…

PolyData (0x7fb92a345ba0)
  N Cells:    17373
  N Points:   8853
  N Strips:   0
  X Bounds:   -4.708e+01, -3.400e+00
  Y Bounds:   -5.731e+01, 2.350e+01
  Z Bounds:   -4.330e+01, -1.177e+01
  N Arrays:   1